# Phase 4: Monte Carlo Playoff Simulation

Brief overview: projecting 2026-27 team strength from blended 
historical performance and current-roster playoff experience, building 
playoff seeding, and simulating the postseason bracket thousands of 
times to estimate championship odds per team.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib

## 2. Load Phase 2/3 Outputs

In [2]:
# loading team season feats from phase 2, and the trained model from phase 3
games_regular_season_grouped = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\team_season_features.csv')
playoff_experience_snapshot = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\playoff_experience_snapshot.csv')
log_reg = joblib.load(r'D:\Data Science Projects\nba-26-27-predictor\models\logistic_regression_model.pkl')

print(games_regular_season_grouped.shape)
print(playoff_experience_snapshot.shape)

(60, 15)
(30, 2)


## 3. Blend Historical Seasons into a Projected Strength Score

**Question:** How do we combine 2024-25 and 2025-26 team performance 
into a single projected strength score for 2026-27, weighting recent 
performance more heavily?

We weight 2025-26 at 70% and 2024-25 at 30% — recent form is more 
predictive of near-future performance than older data, consistent with 
our project's original recency-over-sample-size design choice. This 
reflects franchise-level performance continuity; it does not account 
for individual player movement beyond the playoff experience feature 
(a scoping decision appropriate to a 2-season, team-level dataset, not 
an oversight).

In [3]:
# pivoting so each team has one row w/ separate columns for each seasons feats
strength_cols = ['WIN_PERCENTAGE', 'eFG%', 'TOV%', 'ORB%', 'FT Rate', 'NET_RATING', 
                  'mvp_share', 'ALL_NBA_COUNT']

season_2425 = games_regular_season_grouped[games_regular_season_grouped['SEASON_LABEL'] == '2024-25'].set_index('TEAM_ABBREVIATION')[strength_cols]
season_2526 = games_regular_season_grouped[games_regular_season_grouped['SEASON_LABEL'] == '2025-26'].set_index('TEAM_ABBREVIATION')[strength_cols]

# weighted blend: 70/30 
RECENT_WEIGHT = 0.70
PRIOR_WEIGHT = 0.30

projected_strength = (season_2526 * RECENT_WEIGHT) + (season_2425 * PRIOR_WEIGHT)
projected_strength = projected_strength.reset_index()



## 4. Layer in Roster-Aware Playoff Experience

**Question:** How do we incorporate the one feature that already 
reflects current 2026-27 rosters, rather than relying purely on 
franchise-level historical continuity?

In [4]:
# merging in the current-roster playoff experience snapshot
projected_strength = pd.merge(
    projected_strength, 
    playoff_experience_snapshot, 
    on='TEAM_ABBREVIATION',
    how='left'
)

display(projected_strength.head())

,TEAM_ABBREVIATION,WIN_PERCENTAGE,eFG%,TOV%,ORB%,FT Rate,NET_RATING,mvp_share,ALL_NBA_COUNT,TOTAL_PLAYOFF_EXPERIENCE
0,ATL,0.539024,0.552359,0.119197,0.247925,0.187537,1.362773,0.0000,0.7,141.0
1,BKN,0.265854,0.519659,0.133566,0.237201,0.208118,-9.193637,0.0000,0.0,60.0
2,BOS,0.701220,0.556042,0.104160,0.280003,0.170747,8.413683,0.1556,1.0,196.0
3,CHA,0.445122,0.538638,0.129093,0.293192,0.193813,0.725640,0.0000,0.0,56.0
4,CHI,0.407317,0.549562,0.125400,0.227674,0.187286,-3.949391,0.0000,0.0,33.0


## 5. Compute Win Probability Against a League-Average Opponent

**Question:** How do we turn each team's projected features into a 
single, comparable strength score using the model we already trained?

Rather than inventing a new arbitrary weighting formula, we reuse our 
trained Logistic Regression model — feeding each team's projected 
features against a "league-average opponent" row gives us a model-based 
win probability, which doubles as a ranking metric for seeding.

In [5]:
# building a league-average opponent row serving as a constant baseline
league_avg = projected_strength[strength_cols].mean()

# building the model input, each teams own feats + the league avg
model_input_rows = []

for _, row in projected_strength.iterrows():
    model_row = {
        'WIN_PERCENTAGE': row['WIN_PERCENTAGE'],
        'eFG%': row['eFG%'],
        'TOV%': row['TOV%'],
        'ORB%': row['ORB%'],
        'FT Rate': row['FT Rate'],
        'NET_RATING': row['NET_RATING'],
        'mvp_share': row['mvp_share'],
        'ALL_NBA_COUNT': row['ALL_NBA_COUNT'],
        'OPP_WIN_PERCENTAGE': league_avg['WIN_PERCENTAGE'],
        'OPP_eFG%': league_avg['eFG%'],
        'OPP_TOV%': league_avg['TOV%'],
        'OPP_ORB%': league_avg['ORB%'],
        'OPP_FT Rate': league_avg['FT Rate'],
        'OPP_NET_RATING': league_avg['NET_RATING'],
        'OPP_mvp_share': league_avg['mvp_share'],
        'OPP_ALL_NBA_COUNT': league_avg['ALL_NBA_COUNT']
    }
    model_input_rows.append(model_row)

model_input_df = pd.DataFrame(model_input_rows)

# predicting win prob against a league-average opponent for every team
projected_strength['POWER_SCORE'] = log_reg.predict_proba(model_input_df)[:, 1]
display(projected_strength[['TEAM_ABBREVIATION', 'POWER_SCORE']].sort_values('POWER_SCORE', ascending=False))



,TEAM_ABBREVIATION,POWER_SCORE
20,OKC,0.814463
2,BOS,0.732384
8,DET,0.693994
5,CLE,0.680148
19,NYK,0.668553
7,DEN,0.665096
26,SAS,0.656534
10,HOU,0.645317
17,MIN,0.604684
13,LAL,0.586521


## 6. Incorporate Roster Value into Power Score

**Question:** How do we combine our model-based POWER_SCORE (from 
historical franchise performance) with TOTAL_ROSTER_VALUE (from current, 
roster-aware individual player performance) into one final ranking that 
reflects real 2026-27 team strength?

In [6]:
# loading roster value snapshot 
roster_value_snapshot = pd.read_csv(r'D:\Data Science Projects\nba-26-27-predictor\data\processed\roster_value_snapshot.csv')


# merging onto projected_strength
projected_strength = pd.merge(
    projected_strength,
    roster_value_snapshot[['TEAM_ABBREVIATION', 'TOTAL_ROSTER_VALUE']],
    on='TEAM_ABBREVIATION',
    how='left'
)

display(projected_strength[['TEAM_ABBREVIATION', 'POWER_SCORE', 'TOTAL_ROSTER_VALUE']].head())

,TEAM_ABBREVIATION,POWER_SCORE,TOTAL_ROSTER_VALUE
0,ATL,0.543102,57.010759
1,BKN,0.245296,34.459877
2,BOS,0.732384,50.081889
3,CHA,0.487628,47.352670
4,CHI,0.380195,41.161522


In [7]:
# normalizing playoff experience the same way as roster value
projected_strength['PLAYOFF_EXP_NORMALIZED'] = projected_strength['TOTAL_PLAYOFF_EXPERIENCE'].rank(pct=True)
projected_strength['ROSTER_VALUE_NORMALIZED'] = projected_strength['TOTAL_ROSTER_VALUE'].rank(pct=True)

# final blended power score: 40% model, 45% roster value, 15% playoff experience
FINAL_POWER_WEIGHT_HISTORICAL = 0.40
FINAL_POWER_WEIGHT_ROSTER = 0.45
FINAL_POWER_WEIGHT_PLAYOFF_EXP = 0.15

projected_strength['FINAL_POWER_SCORE'] = (
    projected_strength['POWER_SCORE'] * FINAL_POWER_WEIGHT_HISTORICAL +
    projected_strength['ROSTER_VALUE_NORMALIZED'] * FINAL_POWER_WEIGHT_ROSTER +
    projected_strength['PLAYOFF_EXP_NORMALIZED'] * FINAL_POWER_WEIGHT_PLAYOFF_EXP
)

display(projected_strength[['TEAM_ABBREVIATION', 'FINAL_POWER_SCORE']].sort_values('FINAL_POWER_SCORE', ascending=False))

,TEAM_ABBREVIATION,FINAL_POWER_SCORE
19,NYK,0.802421
7,DEN,0.801038
20,OKC,0.790785
22,PHI,0.737505
26,SAS,0.732613
10,HOU,0.728127
24,POR,0.662005
21,ORL,0.654868
17,MIN,0.611874
5,CLE,0.602059


## 7. Monte Carlo Simulation: Bracket Setup

**Question:** How do we convert these power scores into championship odds?

We'll assign teams to East and West conferences, seed the top 8 in each based on `FINAL_POWER_SCORE`, and simulate a standard NBA playoff bracket 10,000 times. For each series matchup, we will use our trained Logistic Regression model to calculate the probability of Team A beating Team B in a single game, and simulate a Best-of-7 series.

In [8]:
east_teams = ['BOS', 'NYK', 'MIL', 'CLE', 'ORL', 'IND', 'PHI', 'MIA', 'CHI', 'ATL', 'BKN', 'TOR', 'CHA', 'WAS', 'DET']
west_teams = ['OKC', 'DEN', 'MIN', 'LAC', 'DAL', 'PHX', 'NOP', 'LAL', 'SAC', 'GSW', 'HOU', 'UTA', 'MEM', 'SAS', 'POR']

projected_strength['CONFERENCE'] = projected_strength['TEAM_ABBREVIATION'].apply(lambda x: 'East' if x in east_teams else 'West')

# Get top 8 seeds per conference based on FINAL_POWER_SCORE
east_playoff_teams = projected_strength[projected_strength['CONFERENCE'] == 'East'].sort_values('FINAL_POWER_SCORE', ascending=False).head(8)
west_playoff_teams = projected_strength[projected_strength['CONFERENCE'] == 'West'].sort_values('FINAL_POWER_SCORE', ascending=False).head(8)

display(east_playoff_teams[['TEAM_ABBREVIATION', 'FINAL_POWER_SCORE']])
display(west_playoff_teams[['TEAM_ABBREVIATION', 'FINAL_POWER_SCORE']])

,TEAM_ABBREVIATION,FINAL_POWER_SCORE
19,NYK,0.802421
22,PHI,0.737505
21,ORL,0.654868
5,CLE,0.602059
11,IND,0.569752
0,ATL,0.564741
2,BOS,0.537954
27,TOR,0.489052


,TEAM_ABBREVIATION,FINAL_POWER_SCORE
7,DEN,0.801038
20,OKC,0.790785
26,SAS,0.732613
10,HOU,0.728127
24,POR,0.662005
17,MIN,0.611874
13,LAL,0.579608
18,NOP,0.552963


In [9]:
import random

matchup_cache = {}

def get_matchup_prob(team_a_abbrev, team_b_abbrev, df):
    cache_key = f"{team_a_abbrev}_{team_b_abbrev}"
    if cache_key in matchup_cache:
        return matchup_cache[cache_key]
    
    pa = df.loc[df['TEAM_ABBREVIATION'] == team_a_abbrev, 'FINAL_POWER_SCORE'].iloc[0]
    pb = df.loc[df['TEAM_ABBREVIATION'] == team_b_abbrev, 'FINAL_POWER_SCORE'].iloc[0]
    
    prob_a_wins = (pa * (1 - pb)) / ((pa * (1 - pb)) + (pb * (1 - pa)))
    
    matchup_cache[cache_key] = prob_a_wins
    return prob_a_wins

def simulate_series(prob_a_wins):
    # Simulates a Best-of-7 series. Returns True if Team A wins (4 games), False if Team B wins.
    wins_a = 0
    wins_b = 0
    while wins_a < 4 and wins_b < 4:
        if random.random() < prob_a_wins:
            wins_a += 1
        else:
            wins_b += 1
    return wins_a == 4

def play_round(matchups, df):
    winners = []
    for (team_a, team_b) in matchups:
        prob = get_matchup_prob(team_a, team_b, df)
        if simulate_series(prob):
            winners.append(team_a)
        else:
            winners.append(team_b)
    return winners

In [10]:
NUM_SIMULATIONS = 10000
championships = {team: 0 for team in projected_strength['TEAM_ABBREVIATION']}

east_seeds = east_playoff_teams['TEAM_ABBREVIATION'].tolist()
west_seeds = west_playoff_teams['TEAM_ABBREVIATION'].tolist()

for sim in range(NUM_SIMULATIONS):
    # Round 1 Matchups
    east_r1 = [(east_seeds[0], east_seeds[7]), (east_seeds[3], east_seeds[4]), (east_seeds[2], east_seeds[5]), (east_seeds[1], east_seeds[6])]
    west_r1 = [(west_seeds[0], west_seeds[7]), (west_seeds[3], west_seeds[4]), (west_seeds[2], west_seeds[5]), (west_seeds[1], west_seeds[6])]
    
    # Round 1
    east_r2_teams = play_round(east_r1, projected_strength)
    west_r2_teams = play_round(west_r1, projected_strength)

    # Round 2 Matchups — pair up Round 1's winners 
    # (1v8 winner vs 4v5 winner, 3v6 winner vs 2v7 winner)
    east_r2 = [(east_r2_teams[0], east_r2_teams[1]), (east_r2_teams[2], east_r2_teams[3])]
    west_r2 = [(west_r2_teams[0], west_r2_teams[1]), (west_r2_teams[2], west_r2_teams[3])]

    # Round 2
    east_r3_teams = play_round(east_r2, projected_strength)
    west_r3_teams = play_round(west_r2, projected_strength)

    # Conference Finals
    east_champ = play_round([(east_r3_teams[0], east_r3_teams[1])], projected_strength)[0]
    west_champ = play_round([(west_r3_teams[0], west_r3_teams[1])], projected_strength)[0]

    # NBA Finals
    nba_champ = play_round([(east_champ, west_champ)], projected_strength)[0]
    
    championships[nba_champ] += 1

results_df = pd.DataFrame(list(championships.items()), columns=['TEAM', 'CHAMPIONSHIP_WINS'])
results_df['CHAMPIONSHIP_PROBABILITY'] = results_df['CHAMPIONSHIP_WINS'] / NUM_SIMULATIONS
results_df = results_df.sort_values('CHAMPIONSHIP_PROBABILITY', ascending=False)

print(f"--- 2026-27 CHAMPIONSHIP PROBABILITIES (After {NUM_SIMULATIONS} Simulations) ---")
display(results_df[results_df['CHAMPIONSHIP_PROBABILITY'] > 0].reset_index(drop=True).style.format({'CHAMPIONSHIP_PROBABILITY': '{:.2%}'}))

--- 2026-27 CHAMPIONSHIP PROBABILITIES (After 10000 Simulations) ---

,TEAM,CHAMPIONSHIP_WINS,CHAMPIONSHIP_PROBABILITY
0,NYK,3867,38.67%
1,DEN,2438,24.38%
2,OKC,1909,19.09%
3,PHI,944,9.44%
4,SAS,387,3.87%
5,HOU,281,2.81%
6,ORL,94,0.94%
7,POR,38,0.38%
8,CLE,18,0.18%
9,MIN,8,0.08%


In [11]:
import inspect
print(inspect.getsource(get_matchup_prob))

def get_matchup_prob(team_a_abbrev, team_b_abbrev, df):
    cache_key = f"{team_a_abbrev}_{team_b_abbrev}"
    if cache_key in matchup_cache:
        return matchup_cache[cache_key]

    pa = df.loc[df['TEAM_ABBREVIATION'] == team_a_abbrev, 'FINAL_POWER_SCORE'].iloc[0]
    pb = df.loc[df['TEAM_ABBREVIATION'] == team_b_abbrev, 'FINAL_POWER_SCORE'].iloc[0]

    prob_a_wins = (pa * (1 - pb)) / ((pa * (1 - pb)) + (pb * (1 - pa)))

    matchup_cache[cache_key] = prob_a_wins
    return prob_a_wins

